# Categorizing Humor

Set `INPUT_JSON_PATH` to the `laughter_with_text.json` file you want to categorize.


In [ ]:
from google.colab import userdata
import os
import subprocess
import sys
from pathlib import Path

INPUT_JSON_PATH = Path("/content/laughter_with_text.json")

# Model and cost settings for the categorization run.
MODEL_NAME = "gpt-5-nano"
OUTPUT_TOKEN_ESTIMATE_PER_SEGMENT = 900
MODEL_PRICING_USD_PER_1M = {
    "gpt-5.5": {"input": 5.00, "cached_input": 0.50, "output": 30.00},
    "gpt-5.4": {"input": 2.50, "cached_input": 0.25, "output": 15.00},
    "gpt-5.4-mini": {"input": 0.75, "cached_input": 0.075, "output": 4.50},
    "gpt-5.4-nano": {"input": 0.20, "cached_input": 0.02, "output": 1.25},
    "gpt-5": {"input": 1.25, "cached_input": 0.125, "output": 10.00},
    "gpt-5-mini": {"input": 0.25, "cached_input": 0.025, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "cached_input": 0.005, "output": 0.40},
}

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])

try:
    from openai import OpenAI
except ImportError:
    pip_install(["openai"])
    from openai import OpenAI

try:
    from pydantic import BaseModel, Field
except ImportError:
    pip_install(["pydantic"])
    from pydantic import BaseModel, Field

try:
    import tiktoken
except ImportError:
    try:
        pip_install(["tiktoken"])
        import tiktoken
    except Exception as exc:
        tiktoken = None
        print(f"tiktoken unavailable; token estimates will use a character fallback. ({exc})")

api_key = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found in Colab Secrets or environment.")

os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)
print("API key loaded successfully.")


In [ ]:
"""
Categorize the humor source of each laughter segment's text using the OpenAI API.

Input:
  INPUT_JSON_PATH

Output:
  laughter_with_humor_categories.json next to the input file
"""

import json
from time import perf_counter

laughter_json_path = INPUT_JSON_PATH.resolve()
if not laughter_json_path.exists():
    raise FileNotFoundError(f"Could not find {laughter_json_path}.")

with open(laughter_json_path, encoding="utf-8") as f:
    data = json.load(f)

out_path = laughter_json_path.with_name("laughter_with_humor_categories.json")

class HumorCategory(BaseModel):
    label: str
    confidence: float = Field(ge=0.0, le=1.0)
    evidence: str
    reasoning: str

class ScriptOpposition(BaseModel):
    detected: bool
    script_a: str | None = None
    script_b: str | None = None
    opposition_type: str | None = None
    trigger: str | None = None
    trigger_location: str | None = None
    setup_reading: str | None = None
    punch_reinterpretation: str | None = None
    ambiguity_type: str | None = None
    overlap_explanation: str | None = None
    confidence: float = Field(ge=0.0, le=1.0)

class HumorAnalysis(BaseModel):
    input_text: str
    no_humor_detected: bool
    categories: list[HumorCategory] = Field(default_factory=list)
    primary_category: str | None = None
    script_opposition: ScriptOpposition
    overall_explanation: str

SYSTEM_MESSAGE = "You are an expert comedy analyst."
HUMOR_CATEGORIZATION_PROMPT = """You are an expert humor analyst.

Analyze the humor in one structured response. Return both:
1. humor category classification
2. the script-opposition mechanism, if one is present

Use only these exact English humor labels:

- Stupidity: Humor based on ignorance, foolish action, or deliberate playing dumb. It can make the audience feel superior to the foolish person or persona.
- Self-mockery: Humor where the speaker makes fun of themselves. It lowers the self but can also signal insight, courage, or control over criticism.
- Primitive humor: Humor based on taboo, scatology, slapstick, etiquette-breaking, or basic bodily/social instincts, usually without real harm.
- Black humor: Humor that makes light of tragic, fearful, or serious subjects such as death, war, illness, disaster, or powerlessness.
- Irony: Humor based on saying or presenting the opposite of what is meant, often through playful or intellectual reversal rather than direct attack.
- Schadenfreude: Humor from another person's misfortune, clumsiness, embarrassment, or failure, often with relief that it is not happening to oneself.
- Language humor: Humor based on wordplay, puns, ambiguity, grammar, definitions, double meanings, or playful shifts in wording.
- Exaggeration: Humor that stretches reality, scale, logic, or intensity to absurd lengths in order to highlight a feature or situation.
- Understatement: Humor that downplays something intense, emotional, dangerous, or significant by leaving out the expected reaction.
- Clever observation: Observational humor that questions everyday norms or points out unnoticed absurdities in ordinary human behavior.
- Sudden twist: Humor based on setting up an expectation and then sharply violating it, including rule-of-three pattern breaks.
- Inappropriate remark: Humor from breaking norms of politeness, etiquette, tact, or social acceptability by saying the wrong, rude, or shocking thing.
- Circular humor: Humor based on paradox, circular logic, contradiction, self-reference, or asserting something by denying it.
- Anti-humor: Humor from intentionally bad jokes, non sequiturs, missing punchlines, flatness, awkwardness, or subverting normal joke structure.

Use context_long, context, and text together when assigning labels. context_long is the broader setup window and should always be considered first; context is the short local setup right before the text.
Use multi-label classification when needed. Prefer precision over recall.
For each category, provide confidence, evidence, and concise reasoning.

For the script_opposition field, use a Raskin-style semantic analysis:
- Script A is the normal, common-sense, setup-compatible reading activated by the context.
- Script B is the competing reading activated by the laugh-triggering text, punchline, ambiguity, implication, or social move.
- The scripts should overlap in the same text but be opposed or meaningfully distinct.
- Useful opposition types include actual/non-actual, true/false, literal/figurative, serious/trivial, normal/abnormal, possible/impossible, polite/impolite, high status/low status, safe/dangerous, good/bad, life/death, non-sex/sex, money/non-money, and expected/unexpected.
- The trigger is the specific word, phrase, ambiguity, implication, or social move that makes Script B available.
- trigger_location should briefly locate the trigger, for example context, text, final phrase, punchline, or a short quoted fragment.
- setup_reading explains how the audience understands the setup before the turn.
- punch_reinterpretation explains how the audience reinterprets the setup after the trigger.
- ambiguity_type should name the mechanism if relevant, for example lexical ambiguity, syntactic ambiguity, literalization, presupposition shift, role reversal, social norm violation, category shift, irony, taboo shift, or none/unclear.
- overlap_explanation explains why both scripts can fit the same text.
- If there is no clear script opposition, set detected to false, confidence low, and leave script-specific fields null where appropriate.

Do not invent a clean script opposition for noisy transcripts, purely delivery-based laughter, applause, or weak/nonverbal humor. Do not expose hidden chain-of-thought; provide concise analytical explanations only.

Long context:
{context_long}

Context:
{context}

Text:
{text}
"""

source_segments = data.get("segments", [])
total_segments = len(source_segments)
base_output = {k: v for k, v in data.items() if k != "segments"}
run_started_at = perf_counter()
json_file_size_bytes = laughter_json_path.stat().st_size

usage_totals = {
    "api_call_attempt_count": 0,
    "api_call_count": 0,
    "missing_usage_count": 0,
    "input_tokens": 0,
    "cached_input_tokens": 0,
    "output_tokens": 0,
    "output_reasoning_tokens": 0,
    "total_tokens": 0,
    "actual_cost_usd": 0.0,
}


def pricing_key_for_model(model_name):
    if model_name in MODEL_PRICING_USD_PER_1M:
        return model_name
    for known_model in sorted(MODEL_PRICING_USD_PER_1M, key=len, reverse=True):
        if model_name.startswith(f"{known_model}-"):
            return known_model
    supported = ", ".join(sorted(MODEL_PRICING_USD_PER_1M))
    raise ValueError(
        f"No pricing configured for model {model_name!r}. "
        f"Add it to MODEL_PRICING_USD_PER_1M before running. Supported models: {supported}"
    )


def pricing_for_model(model_name):
    return MODEL_PRICING_USD_PER_1M[pricing_key_for_model(model_name)]


def estimate_cost_usd(input_tokens, output_tokens, cached_input_tokens=0, model_name=MODEL_NAME):
    prices = pricing_for_model(model_name)
    input_tokens = int(input_tokens or 0)
    output_tokens = int(output_tokens or 0)
    cached_input_tokens = max(0, min(int(cached_input_tokens or 0), input_tokens))
    uncached_input_tokens = max(input_tokens - cached_input_tokens, 0)
    return (
        uncached_input_tokens / 1_000_000 * prices["input"]
        + cached_input_tokens / 1_000_000 * prices["cached_input"]
        + output_tokens / 1_000_000 * prices["output"]
    )


def format_bytes(num_bytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if num_bytes < 1024 or unit == "GB":
            return f"{num_bytes:.1f} {unit}" if unit != "B" else f"{num_bytes} {unit}"
        num_bytes /= 1024


def format_cost(value, places=4):
    return f"${value:,.{places}f}"


def format_cost_compact(value):
    value = float(value or 0.0)
    places = 6 if abs(value) < 0.001 else 4 if abs(value) < 1 else 2
    return format_cost(value, places=places)


def build_humor_prompt(context_long, context, text):
    return (
        HUMOR_CATEGORIZATION_PROMPT
        .replace("{context_long}", context_long)
        .replace("{context}", context)
        .replace("{text}", text)
    )


def build_input_messages(context_long, context, text):
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": build_humor_prompt(context_long, context, text)},
    ]


def token_encoder_for_model(model_name):
    if tiktoken is None:
        return None
    try:
        return tiktoken.encoding_for_model(model_name)
    except Exception:
        try:
            return tiktoken.get_encoding("o200k_base")
        except Exception:
            return None


TOKEN_ENCODER = token_encoder_for_model(MODEL_NAME)


def estimate_text_tokens(text):
    text = str(text or "")
    if TOKEN_ENCODER is not None:
        return len(TOKEN_ENCODER.encode(text))
    return max(1, (len(text) + 3) // 4)


def estimate_input_message_tokens(messages):
    token_count = 0
    for message in messages:
        token_count += 8  # small chat/message framing allowance
        token_count += estimate_text_tokens(message.get("role", ""))
        token_count += estimate_text_tokens(message.get("content", ""))
    return token_count


def segment_text(seg):
    return (seg.get("text") or "").strip()


def segment_contexts(seg):
    context = (seg.get("context") or "").strip()
    context_long = (seg.get("context_long") or "").strip()
    if not context_long:
        context_long = context
    return context_long, context


def estimate_segment_input_tokens(seg):
    text = segment_text(seg)
    if not text:
        return 0
    context_long, context = segment_contexts(seg)
    return estimate_input_message_tokens(build_input_messages(context_long, context, text))


def get_value(obj, name, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def first_present(*values, default=0):
    for value in values:
        if value is not None:
            return value
    return default


def to_int(value):
    try:
        return int(value or 0)
    except (TypeError, ValueError):
        return 0


def usage_from_response(response):
    usage = get_value(response, "usage")
    response_model = get_value(response, "model", MODEL_NAME) or MODEL_NAME
    pricing_model = pricing_key_for_model(MODEL_NAME)

    if usage is None:
        return {
            "model": response_model,
            "pricing_model": pricing_model,
            "usage_available": False,
            "input_tokens": 0,
            "cached_input_tokens": 0,
            "uncached_input_tokens": 0,
            "output_tokens": 0,
            "output_reasoning_tokens": 0,
            "total_tokens": 0,
            "estimated_cost_usd": 0.0,
        }

    input_details = first_present(
        get_value(usage, "input_tokens_details"),
        get_value(usage, "prompt_tokens_details"),
        default={},
    )
    output_details = first_present(get_value(usage, "output_tokens_details"), default={})

    input_tokens = to_int(first_present(get_value(usage, "input_tokens"), get_value(usage, "prompt_tokens")))
    output_tokens = to_int(first_present(get_value(usage, "output_tokens"), get_value(usage, "completion_tokens")))
    total_tokens = to_int(first_present(get_value(usage, "total_tokens"), default=input_tokens + output_tokens))
    cached_input_tokens = to_int(first_present(
        get_value(input_details, "cached_tokens"),
        get_value(input_details, "cached_input_tokens"),
        get_value(usage, "cached_input_tokens"),
    ))
    cached_input_tokens = max(0, min(cached_input_tokens, input_tokens))
    output_reasoning_tokens = to_int(first_present(get_value(output_details, "reasoning_tokens")))
    estimated_cost_usd = estimate_cost_usd(
        input_tokens,
        output_tokens,
        cached_input_tokens=cached_input_tokens,
        model_name=MODEL_NAME,
    )

    return {
        "model": response_model,
        "pricing_model": pricing_model,
        "usage_available": True,
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_input_tokens,
        "uncached_input_tokens": max(input_tokens - cached_input_tokens, 0),
        "output_tokens": output_tokens,
        "output_reasoning_tokens": output_reasoning_tokens,
        "total_tokens": total_tokens,
        "estimated_cost_usd": estimated_cost_usd,
    }


def add_usage(usage_summary):
    usage_totals["api_call_count"] += 1
    if not usage_summary.get("usage_available", True):
        usage_totals["missing_usage_count"] += 1
    for key in [
        "input_tokens",
        "cached_input_tokens",
        "output_tokens",
        "output_reasoning_tokens",
        "total_tokens",
    ]:
        usage_totals[key] += int(usage_summary.get(key, 0) or 0)
    usage_totals["actual_cost_usd"] += float(usage_summary.get("estimated_cost_usd", 0.0) or 0.0)


pricing_key_for_model(MODEL_NAME)
estimated_input_tokens_by_position = [estimate_segment_input_tokens(seg) for seg in source_segments]
segments_requiring_api = sum(1 for tokens in estimated_input_tokens_by_position if tokens > 0)
total_estimated_input_tokens = sum(estimated_input_tokens_by_position)
initial_estimated_output_tokens = segments_requiring_api * OUTPUT_TOKEN_ESTIMATE_PER_SEGMENT
initial_estimated_cost_usd = estimate_cost_usd(total_estimated_input_tokens, initial_estimated_output_tokens)


def build_usage_summary(processed_segments):
    processed_count = len(processed_segments)
    remaining_input_tokens = sum(estimated_input_tokens_by_position[processed_count:])
    remaining_api_calls = sum(1 for tokens in estimated_input_tokens_by_position[processed_count:] if tokens > 0)

    if usage_totals["api_call_count"]:
        average_output_tokens = usage_totals["output_tokens"] / usage_totals["api_call_count"]
    else:
        average_output_tokens = OUTPUT_TOKEN_ESTIMATE_PER_SEGMENT

    if usage_totals["input_tokens"]:
        cached_input_ratio = usage_totals["cached_input_tokens"] / usage_totals["input_tokens"]
    else:
        cached_input_ratio = 0.0

    estimated_remaining_output_tokens = int(round(remaining_api_calls * average_output_tokens))
    estimated_remaining_cached_input_tokens = int(round(remaining_input_tokens * cached_input_ratio))
    estimated_remaining_cost_usd = estimate_cost_usd(
        remaining_input_tokens,
        estimated_remaining_output_tokens,
        cached_input_tokens=estimated_remaining_cached_input_tokens,
    )
    failed_count = sum(1 for seg in processed_segments if seg.get("humor_error"))
    empty_count = sum(1 for seg in processed_segments if not segment_text(seg))

    return {
        "model": MODEL_NAME,
        "pricing_model": pricing_key_for_model(MODEL_NAME),
        "pricing_usd_per_1m": pricing_for_model(MODEL_NAME),
        "json_file_size_bytes": json_file_size_bytes,
        "total_segments": total_segments,
        "processed_segments": processed_count,
        "segments_requiring_api": segments_requiring_api,
        "empty_segments_processed": empty_count,
        "failed_segments": failed_count,
        "api_call_attempt_count": usage_totals["api_call_attempt_count"],
        "api_call_count": usage_totals["api_call_count"],
        "missing_usage_count": usage_totals["missing_usage_count"],
        "input_tokens": usage_totals["input_tokens"],
        "cached_input_tokens": usage_totals["cached_input_tokens"],
        "uncached_input_tokens": max(usage_totals["input_tokens"] - usage_totals["cached_input_tokens"], 0),
        "output_tokens": usage_totals["output_tokens"],
        "output_reasoning_tokens": usage_totals["output_reasoning_tokens"],
        "total_tokens": usage_totals["total_tokens"],
        "actual_cost_usd": usage_totals["actual_cost_usd"],
        "estimated_initial_input_tokens": total_estimated_input_tokens,
        "estimated_initial_output_tokens": initial_estimated_output_tokens,
        "estimated_initial_cost_usd": initial_estimated_cost_usd,
        "estimated_remaining_input_tokens": remaining_input_tokens,
        "estimated_remaining_output_tokens": estimated_remaining_output_tokens,
        "estimated_remaining_cost_usd": estimated_remaining_cost_usd,
        "projected_final_cost_usd": usage_totals["actual_cost_usd"] + estimated_remaining_cost_usd,
        "duration_seconds": perf_counter() - run_started_at,
        "completed": processed_count == total_segments,
    }


def format_usage_status(processed_segments):
    summary = build_usage_summary(processed_segments)
    return (
        f"tokens {summary['input_tokens']:,} in / {summary['output_tokens']:,} out | "
        f"cost {format_cost_compact(summary['actual_cost_usd'])} | "
        f"projected {format_cost_compact(summary['projected_final_cost_usd'])}"
    )


def write_progress(processed_segments):
    payload = {
        **base_output,
        "segments": processed_segments,
        "processed_count": len(processed_segments),
        "total_segments": total_segments,
        "completed": len(processed_segments) == total_segments,
        "usage_summary": build_usage_summary(processed_segments),
    }
    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp_path.replace(out_path)


def clear_script_opposition_fields(seg):
    seg["script_opposition"] = None
    seg["script_opposition_detected"] = None
    seg["script_a"] = None
    seg["script_b"] = None
    seg["opposition_type"] = None
    seg["humor_trigger"] = None
    seg["trigger_location"] = None
    seg["setup_reading"] = None
    seg["punch_reinterpretation"] = None
    seg["ambiguity_type"] = None
    seg["overlap_explanation"] = None
    seg["script_opposition_confidence"] = None


def attach_script_opposition_fields(seg, script_opposition):
    seg["script_opposition"] = script_opposition.model_dump()
    seg["script_opposition_detected"] = script_opposition.detected
    seg["script_a"] = script_opposition.script_a
    seg["script_b"] = script_opposition.script_b
    seg["opposition_type"] = script_opposition.opposition_type
    seg["humor_trigger"] = script_opposition.trigger
    seg["trigger_location"] = script_opposition.trigger_location
    seg["setup_reading"] = script_opposition.setup_reading
    seg["punch_reinterpretation"] = script_opposition.punch_reinterpretation
    seg["ambiguity_type"] = script_opposition.ambiguity_type
    seg["overlap_explanation"] = script_opposition.overlap_explanation
    seg["script_opposition_confidence"] = script_opposition.confidence


print(
    f"Loaded {total_segments:,} segments from {laughter_json_path.name} "
    f"({format_bytes(json_file_size_bytes)}); {segments_requiring_api:,} contain text."
)
print(
    f"Using {MODEL_NAME}; initial estimate: "
    f"{total_estimated_input_tokens:,} input tokens + "
    f"{initial_estimated_output_tokens:,} output tokens = "
    f"{format_cost_compact(initial_estimated_cost_usd)}."
)

write_progress([])
print(f"Initialized {out_path} (0/{total_segments} segments processed) | {format_usage_status([])}")

results = []
for seg in source_segments:
    seg = dict(seg)
    text = segment_text(seg)
    seg.pop("humor_error", None)
    seg.pop("usage", None)

    if not text:
        seg["humor_structured"] = None
        seg["humor_category"] = None
        seg["humor_explanation"] = None
        clear_script_opposition_fields(seg)
        results.append(seg)
        write_progress(results)
        print(f"Segment {seg.get('index')}: no text ({len(results)}/{total_segments}) | {format_usage_status(results)}")
        continue

    try:
        context_long, context = segment_contexts(seg)
        usage_totals["api_call_attempt_count"] += 1
        response = client.responses.parse(
            model=MODEL_NAME,
            input=build_input_messages(context_long, context, text),
            text_format=HumorAnalysis,
        )

        parsed: HumorAnalysis = response.output_parsed
        usage_summary = usage_from_response(response)
        add_usage(usage_summary)
        seg["usage"] = usage_summary
        seg["humor_structured"] = parsed.model_dump()
        seg["humor_category"] = parsed.primary_category
        seg["humor_explanation"] = parsed.overall_explanation
        attach_script_opposition_fields(seg, parsed.script_opposition)

        results.append(seg)
        write_progress(results)
        print(
            f"Segment {seg.get('index')}: {parsed.primary_category} "
            f"({len(results)}/{total_segments}) | {format_usage_status(results)}"
        )

    except Exception as e:
        seg["humor_structured"] = None
        seg["humor_category"] = None
        seg["humor_explanation"] = None
        clear_script_opposition_fields(seg)
        seg["humor_error"] = str(e)
        results.append(seg)
        write_progress(results)
        print(f"Segment {seg.get('index')}: error -> {e} ({len(results)}/{total_segments}) | {format_usage_status(results)}")

final_usage = build_usage_summary(results)
print(f"\nSaved {out_path} with humor categories for {len(results)} segments.")
print(
    f"Final usage: {final_usage['input_tokens']:,} input tokens, "
    f"{final_usage['output_tokens']:,} output tokens, "
    f"{format_cost_compact(final_usage['actual_cost_usd'])} actual estimated cost."
)
